# 00 - LLaMEA Prompt Inspection & Variations

This notebook provides a structured, interactive view of **all prompt components** used in the LLaMEA evolutionary synthesis loop:

1. **Shared Components**:
   - `format.j2`: Strict Python output format, class signatures, and rules.
   - `example.j2`: Starter class skeleton, docstring, and budget-tracking pattern.
2. **The 3 Main Synthesis Modes**:
   - `clean`: Noise-free, deterministic landscape.
   - `implicit`: Unknown real-world black-box landscape.
   - `noisy`: Stochastic landscape with explicit warnings on noise traps.
3. **The 4 Strategy Levels (under Noisy Mode)**:
   - Level 1: `baseline` (zero guidance)
   - Level 2: `vectorization` (NumPy matrix operations & batch sampling)
   - Level 3: `guided` (algorithmic archetypes & starter re-evaluation with k=3)
   - Level 4: `thinking` (Socratic reasoning prompts)
4. **Full Assembled Prompts**: Complete prompt payloads as dispatched to the LLM during generation.


In [14]:
import sys
from pathlib import Path
import numpy as np

# Ensure project root & src are in path
cwd = Path(".").resolve()
root_dir = cwd.parent if cwd.name == "notebooks" else cwd
src_dir = root_dir / "src"
if str(src_dir) not in sys.path:
    sys.path.insert(0, str(src_dir))

from evolution.domain.enums import SynthesisMode, PromptStrategy
from evolution.infra.prompts import (
    build_task_prompt,
    build_example_prompt,
    build_format_prompt,
)
from IPython.display import display, Markdown

# Default sample parameters for inspection
problem_id = 1
dim = 2
lb = np.array([-5.0, -5.0])
ub = np.array([5.0, 5.0])
budget = 2000

print("[OK] LLaMEA Prompt builder loaded successfully.")


[OK] LLaMEA Prompt builder loaded successfully.


---
## 1. Output Format Prompt (`format.j2`)

This prompt is passed to LLaMEA as `output_format_prompt`. It enforces strict syntax rules (e.g. single python code block, class structure, `__call__(self, problem, budget)` signature, and bans pre-built solver wrappers like `scipy.optimize`).


In [15]:
format_prompt = build_format_prompt()
print(format_prompt)


Respond with EXACTLY the following format — no extra code blocks:

Feedback: <your reasoning and description of the algorithm>
Code:
```python
<your complete class and any required imports>
```

STRICT Rules — violating any rule will cause execution failure:
- There must be exactly ONE ```python ... ``` block in your response.
- The class MUST be named exactly one word (e.g., `class MyOptimizer:`).
- `__init__(self)` MUST take NO extra arguments beyond `self`.
- `__init__(self)` MUST have a non-empty body (use `pass` if nothing to initialize).
- The class MUST have a `__call__(self, problem, budget)` method.
- `__call__` MUST return `(best_x, float(best_y))` — a tuple of the best search coordinates array and best scalar float value.
- Do NOT import or call `scipy.optimize` (e.g. `scipy.optimize.minimize`, `differential_evolution`, etc.) — pre-built solver wrappers are strictly banned. Write your search algorithm logic from scratch using NumPy.
- Every variable you use MUST be defined b

---
## 2. Example Code Skeleton Prompt (`example.j2`)

This prompt is passed to LLaMEA as `example_prompt`. It provides the candidate algorithm skeleton, showing how bounds are extracted, how `evaluations` must be incremented, and what tuple must be returned.


In [16]:
example_prompt = build_example_prompt()
print(example_prompt)


Your algorithm will be instantiated and called as follows:
    optimizer = AlgorithmName()
    best_x, best_y = optimizer(problem, budget)

You MUST use the following class skeleton — fill in your algorithm logic in the marked section only.
Do NOT change the class structure, method signatures, or return statement:

    import numpy as np

    class AlgorithmName:
        def __init__(self):
            pass  # Add initialization state here if your algorithm needs it

        def __call__(self, problem, budget):
            lb = np.asarray(getattr(problem, 'lower_bound', -5.0), dtype=float)
            ub = np.asarray(getattr(problem, 'upper_bound', 5.0), dtype=float)
            dim = int(getattr(problem, 'dim', len(lb) if hasattr(lb, '__len__') else 3))

            # Always start with a random initial point using vectorization
            best_x = np.random.uniform(lb, ub, size=dim)
            best_y = float(problem(best_x))
            evaluations = 1

            # --- YOUR ALGORI

---
## 3. The 3 Main Synthesis Modes (Problem Landscape)

The outer task prompt injects different landscape characteristics into `layout.j2`:
- **Clean Mode (`SynthesisMode.CLEAN`)**: Deterministic, noise-free.
- **Implicit Mode (`SynthesisMode.IMPLICIT`)**: Unknown black-box.
- **Noisy Mode (`SynthesisMode.NOISY`)**: Stochastic with warnings about single-shot acceptance, smoothed tracking, and budget leaks.


In [17]:
modes = [SynthesisMode.CLEAN, SynthesisMode.IMPLICIT, SynthesisMode.NOISY]

for mode in modes:
    p = build_task_prompt(
        problem_id=problem_id,
        dim=dim,
        lower_bound=lb,
        upper_bound=ub,
        mode=mode,
        strategy=PromptStrategy.BASELINE,
        budget_hint=budget,
    )
    print("=" * 80)
    print(f"=== SYNTHESIS MODE: {mode.value.upper()} (Baseline Strategy) ===")
    print("=" * 80)
    print(p)
    print("\n")


=== SYNTHESIS MODE: CLEAN (Baseline Strategy) ===
You are a highly skilled computer scientist and an expert in meta-heuristic optimization.
Your task is to design a novel, continuous black-box optimization algorithm specialized for a specific target landscape (BBOB Problem ID: 1).

Landscape Characteristics:
The objective function is entirely deterministic (noise-free). You can rely on precise evaluations, exact gradient approximations, and aggressive local search exploitation.

This is NOT a general-purpose solver. You are designing a bespoke algorithm tailored to exploit the specific features of this single noise-free landscape.

Problem Parameters:
- The search space is 2-dimensional.
- Search Bounds: [[-5.0, -5.0], [5.0, 5.0]] (accessible via `problem.lower_bound` / `problem.upper_bound`)
- Total Evaluation Budget: 2000 calls to problem(x)


=== SYNTHESIS MODE: IMPLICIT (Baseline Strategy) ===
You are a highly skilled computer scientist and an expert in meta-heuristic optimization.

---
## 4. The 4 Strategy Levels for Noisy Problems

Under `SynthesisMode.NOISY`, the synthesis engine injects 4 progressive levels of strategy guidance:
1. **Level 1 (`baseline`)**: Problem description only, no extra guidance.
2. **Level 2 (`vectorization`)**: Suggests population-level NumPy operations and vector re-evaluation.
3. **Level 3 (`guided`)**: Suggests algorithmic families (CMA, SA, Pop) and concrete code with $k=3$ sample averaging.
4. **Level 4 (`thinking`)**: Socratic questions prompting reasoning before coding.


In [18]:
strategies = [
    PromptStrategy.BASELINE,
    PromptStrategy.VECTORIZATION,
    PromptStrategy.GUIDED,
    PromptStrategy.THINKING,
]

for strat in strategies:
    p = build_task_prompt(
        problem_id=problem_id,
        dim=dim,
        lower_bound=lb,
        upper_bound=ub,
        mode=SynthesisMode.NOISY,
        strategy=strat,
        budget_hint=budget,
    )
    print("=" * 80)
    print(f"=== NOISY MODE with STRATEGY: {strat.value.upper()} ===")
    print("=" * 80)
    print(p)
    print("\n")


=== NOISY MODE with STRATEGY: BASELINE ===
You are a highly skilled computer scientist and an expert in meta-heuristic optimization.
Your task is to design a novel, continuous black-box optimization algorithm specialized for a specific target landscape (BBOB Problem ID: 1).

Landscape Characteristics:
The objective function is stochastic — every call to problem(x) returns a different value even at the same point x, due to random noise injected into the true objective value.

A single comparison `if y_new < y_best` may accept or reject candidates based on noise rather than true quality.

Common patterns that silently break under noise:
- Single-shot acceptance: `if trial_y < best_y` -> unreliable; may accept noise artifacts.
- Smoothed tracking: `best_y = 0.9*best_y + 0.1*trial_y` -> corrupts best_y; return becomes invalid.
- Hidden calls: `min(a, b, c, key=lambda x: problem(x))` -> uncounted budget calls per use.
- Misplaced counter: `evaluations += 1` outside the innermost problem() c

---
## 5. Full Assembled Prompt Generator (Ready to Copy)

LLaMEA combines the Task Prompt, Output Format Prompt, and Example Skeleton. Use the helper below to print the exact complete prompt payload for any combination of Mode and Strategy.


In [19]:
def print_full_prompt(mode=SynthesisMode.NOISY, strategy=PromptStrategy.GUIDED, p_id=1, d=2, b=2000):
    task = build_task_prompt(
        problem_id=p_id,
        dim=d,
        lower_bound=np.array([-5.0] * d),
        upper_bound=np.array([5.0] * d),
        mode=mode,
        strategy=strategy,
        budget_hint=b,
    )
    fmt = build_format_prompt()
    ex = build_example_prompt()

    separator = "#" * 80
    output = f"""{separator}
# 1. TASK PROMPT (Problem Description & Strategy Guidance)
{separator}
{task}

{separator}
# 2. OUTPUT FORMAT RULES
{separator}
{fmt}

{separator}
# 3. CODE SKELETON EXAMPLE
{separator}
{ex}
"""
    print(output)
    return output

# Run for Noisy + Guided
prompt_text = print_full_prompt(mode=SynthesisMode.NOISY, strategy=PromptStrategy.GUIDED)


################################################################################
# 1. TASK PROMPT (Problem Description & Strategy Guidance)
################################################################################
You are a highly skilled computer scientist and an expert in meta-heuristic optimization.
Your task is to design a novel, continuous black-box optimization algorithm specialized for a specific target landscape (BBOB Problem ID: 1).

Landscape Characteristics:
The objective function is stochastic — every call to problem(x) returns a different value even at the same point x, due to random noise injected into the true objective value.

A single comparison `if y_new < y_best` may accept or reject candidates based on noise rather than true quality.

Common patterns that silently break under noise:
- Single-shot acceptance: `if trial_y < best_y` -> unreliable; may accept noise artifacts.
- Smoothed tracking: `best_y = 0.9*best_y + 0.1*trial_y` -> corrupts best_y; return beco

---
## 6. Export All Prompts to a Single File (`all_prompts.md`)

This function compiles every single prompt template, strategy level, format rule, and full assembled example into a comprehensive reference document saved as `all_prompts.md`.


In [20]:
def export_all_prompts(dest_path=None):
    if dest_path is None:
        dest_path = root_dir / "docs" / "all_prompts.md"
    else:
        dest_path = Path(dest_path)
    
    dest_path.parent.mkdir(parents=True, exist_ok=True)
    
    dim = 2
    lb = np.array([-5.0] * dim)
    ub = np.array([5.0] * dim)
    budget = 2000

    sections = []
    sections.append("# Complete LLaMEA Prompt Reference Manual\n")
    sections.append("> Comprehensive collection of all prompt components, synthesis modes, strategy levels, and assembled prompts used in the AAD-LLM evolutionary synthesis pipeline.\n")
    
    sections.append("## Table of Contents\n"
                    "- [1. Output Format Rules (`format.j2`)](#1-output-format-rules-formatj2)\n"
                    "- [2. Code Skeleton Example (`example.j2`)](#2-code-skeleton-example-examplej2)\n"
                    "- [3. The 3 Main Synthesis Modes (Landscape Nature)](#3-the-3-main-synthesis-modes-landscape-nature)\n"
                    "  - [3.1 Clean Mode](#31-clean-mode)\n"
                    "  - [3.2 Implicit Mode](#32-implicit-mode)\n"
                    "  - [3.3 Noisy Mode](#33-noisy-mode)\n"
                    "- [4. The 4 Strategy Levels (Noisy Landscape)](#4-the-4-strategy-levels-noisy-landscape)\n"
                    "  - [4.1 Baseline Strategy](#41-baseline-strategy)\n"
                    "  - [4.2 Vectorization Strategy](#42-vectorization-strategy)\n"
                    "  - [4.3 Guided Strategy (k=3)](#43-guided-strategy-k3)\n"
                    "  - [4.4 Thinking Strategy](#44-thinking-strategy)\n"
                    "- [5. Full Assembled Payloads (What LLaMEA Sends to the LLM)](#5-full-assembled-payloads-what-llamea-sends-to-the-llm)\n"
                    "  - [5.1 Clean Baseline](#51-clean-baseline)\n"
                    "  - [5.2 Implicit Baseline](#52-implicit-baseline)\n"
                    "  - [5.3 Noisy Baseline](#53-noisy-baseline)\n"
                    "  - [5.4 Noisy Vectorization](#54-noisy-vectorization)\n"
                    "  - [5.5 Noisy Guided](#55-noisy-guided)\n"
                    "  - [5.6 Noisy Thinking](#56-noisy-thinking)\n\n---\n")
    
    # 1. Format
    sections.append("## 1. Output Format Rules (`format.j2`)\n")
    sections.append("Passed to LLaMEA as `output_format_prompt`:\n")
    sections.append("```text\n" + build_format_prompt().strip() + "\n```\n\n---\n")

    # 2. Example
    sections.append("## 2. Code Skeleton Example (`example.j2`)\n")
    sections.append("Passed to LLaMEA as `example_prompt`:\n")
    sections.append("```python\n" + build_example_prompt().strip() + "\n```\n\n---\n")

    # 3. Modes
    sections.append("## 3. The 3 Main Synthesis Modes (Landscape Nature)\n")
    for mode in [SynthesisMode.CLEAN, SynthesisMode.IMPLICIT, SynthesisMode.NOISY]:
        p = build_task_prompt(problem_id=1, dim=dim, lower_bound=lb, upper_bound=ub, mode=mode, strategy=PromptStrategy.BASELINE, budget_hint=budget)
        sections.append(f"### 3.{[SynthesisMode.CLEAN, SynthesisMode.IMPLICIT, SynthesisMode.NOISY].index(mode) + 1} {mode.value.capitalize()} Mode\n")
        sections.append("```text\n" + p.strip() + "\n```\n")
    sections.append("\n---\n")

    # 4. Strategies
    sections.append("## 4. The 4 Strategy Levels (Noisy Landscape)\n")
    strats = [PromptStrategy.BASELINE, PromptStrategy.VECTORIZATION, PromptStrategy.GUIDED, PromptStrategy.THINKING]
    for idx, strat in enumerate(strats, start=1):
        p = build_task_prompt(problem_id=1, dim=dim, lower_bound=lb, upper_bound=ub, mode=SynthesisMode.NOISY, strategy=strat, budget_hint=budget)
        sections.append(f"### 4.{idx} {strat.value.capitalize()} Strategy\n")
        sections.append("```text\n" + p.strip() + "\n```\n")
    sections.append("\n---\n")

    # 5. Full Assembled
    sections.append("## 5. Full Assembled Payloads (What LLaMEA Sends to the LLM)\n")
    combos = [
        ("Clean Baseline", SynthesisMode.CLEAN, PromptStrategy.BASELINE),
        ("Implicit Baseline", SynthesisMode.IMPLICIT, PromptStrategy.BASELINE),
        ("Noisy Baseline", SynthesisMode.NOISY, PromptStrategy.BASELINE),
        ("Noisy Vectorization", SynthesisMode.NOISY, PromptStrategy.VECTORIZATION),
        ("Noisy Guided", SynthesisMode.NOISY, PromptStrategy.GUIDED),
        ("Noisy Thinking", SynthesisMode.NOISY, PromptStrategy.THINKING),
    ]
    for idx, (title, mode, strat) in enumerate(combos, start=1):
        task = build_task_prompt(problem_id=1, dim=dim, lower_bound=lb, upper_bound=ub, mode=mode, strategy=strat, budget_hint=budget)
        fmt = build_format_prompt()
        ex = build_example_prompt()
        full = f"=== 1. TASK PROMPT ===\n{task.strip()}\n\n=== 2. OUTPUT FORMAT RULES ===\n{fmt.strip()}\n\n=== 3. EXAMPLE CODE SKELETON ===\n{ex.strip()}"
        sections.append(f"### 5.{idx} {title}\n")
        sections.append("```text\n" + full.strip() + "\n```\n")

    content = "\n".join(sections)
    with open(dest_path, "w", encoding="utf-8") as f:
        f.write(content)
    print(f"[OK] Exported all prompts ({len(content):,} chars) to: {dest_path}")
    return dest_path

exported_file = export_all_prompts()


[OK] Exported all prompts (47,896 chars) to: /Users/nicolaibrahim/Desktop/proj/AAD_LLM/docs/all_prompts.md
